# `%catalog` — runs immediately, no commit needed

A worked example of the `%catalog` magic from `eea_datalakehouse.notebook.magics` — see
`docs/notebook-facade-for-data-scientists.md` for the design behind it.

**Before running this for real:** set `DREMIO_BASE_URL`, `DREMIO_TOKEN` (and `DREMIO_USERNAME`
if you'll call `data_copy`/`data_move`) in the kernel environment — the same variables
`debugger/debug_run.py` uses. Install the extra this needs once: `pip install "EEADataLakehouse[notebook]"`.

Nothing below invents new vocabulary: every call after `%catalog` is a real
`CatalogSession` method (`src/eea_datalakehouse/catalog/session.py`) — this notebook is a tour of
that class, not a separate API. Each call runs against Dremio the moment its cell executes;
there's no separate commit step to remember (see `%catalog help`).

In [ ]:
import eea_datalakehouse.notebook  # registers %catalog/%ingest — no %load_ext needed

`%load_ext eea_datalakehouse.notebook.magics` still works too, and is safe to run either
before or after the import above — whichever runs first registers the magics, the other is
a no-op (see `magics.load_ipython_extension`'s docstring).

## Quick reference

`%catalog help` (or `%catalog help()`) renders every command as an HTML table — name,
parameters, description — rather than a raw Python signature, since a data custodian
reading it may not be fluent in Python type-hint syntax. Handy when you don't remember
an exact parameter name mid-notebook. It works even before `DREMIO_BASE_URL`/
`DREMIO_TOKEN` are set, since it's answered before a session is built.

In [ ]:
%catalog help()

## Each call runs right away

There's no queue and no `commit()` to call afterwards — the moment a cell below runs, the
operation has already happened against Dremio (with the same retry-on-cold-engine and
rollback-on-failure safety `commit()` always had, applied automatically per call). The
same `CatalogSession` is still reused across every `%catalog` cell in this kernel — so
context set via `use()` stays in effect across cells, and idempotency keys keep
incrementing — not to batch anything.

In [ ]:
%catalog data_copy("bwd.draft.raw_2026", "bwd.reference.water_temperature", overwrite=True)

In [ ]:
%catalog set_tags("bwd.reference.water_temperature", ["reviewed", "2026"])

## No commit needed

By the time the `set_tags` cell above finished running, both calls had already
happened — `%catalog` commits each one right after it runs (`retry=True` under the hood,
so a step that hits a Dremio engine still warming up is re-attempted automatically).
There's nothing left to flush.

In [ ]:
%catalog get_context()  # -> 'catalog' — its default; data_copy/set_tags only *read* context, never write it; only use() does

### `%%catalog` — set context once for the whole cell

The cell-magic form runs `use(...)` on its magic line, then every other line of the
cell body in order, all under that context — no need to repeat `%catalog` or the path
on each line.

In [ ]:
%%catalog use("bwd.reference")
set_tags("water_temperature", ["archived"])
create_folder("2027")

### `use`'s live existence check

`create_folder`/`set_tags` above dropped the leading `.` on `2027`/`water_temperature`
entirely — once a context exists, every single-path verb's own `path` accepts a bare
name this way (`"2027"` and `".2027"` mean the same thing), and so does `data_copy`/
`data_move`/`create_view`'s own `source_path`/`target_path` (each resolved independently
against that same context — see below). Neither call above changed the context itself,
though — it's still `"bwd.reference"`, exactly what `use` set it to on the magic line
above. Only `use`/`set_context` ever change context; every other verb only *reads* it to
resolve its own `path`. `use` itself is the one real exception to the dot-optional
reading above: its own `path` must always be a whole, absolute path — never resolved
against whatever context already exists, and never a leading `.`/`../` fragment.

A bare path starting with `catalog` (this deployment's one real root source, e.g.
`"catalog.other_root.assessments"`) is also always taken literally as absolute rather
than appended to the context — but that's not an exception to the dot-optional rule
above, just a further refinement of what "bare" means: without it, passing a full
`catalog....` path alongside an already-set context would silently produce a nonsense
double-nested path instead of raising or doing the obvious thing. It's also what makes
`data_copy`/`data_move`/`create_view` safe to resolve this way at all — pairing a
relative `source_path` with a genuinely unrelated absolute `target_path` stays
unambiguous, since a real absolute path here always starts with `catalog`.

`use(None)`/`use("")` reset the context back to `"catalog"` (the root) rather than clearing it entirely — the same place a fresh session already starts. `set_context(None)` is the one way left to clear it to no context at all.

`use` also makes a live check none of the others do: `path` must already exist in the
catalog, or it raises instead of quietly pointing context somewhere later calls would fail
against anyway. That's why the cell below re-enters the full `"bwd.reference.2027"`,
created above, rather than a path nothing has created yet — nothing set context there
automatically, so a fresh `use()` is the only way to actually move.

`use` prints `context set to <path>` itself, rather than an empty commit report — there's
nothing to commit, since `use` never queues anything, so `%catalog` shows the context
instead of a blank result.

In [ ]:
%catalog use("bwd.reference.2027")  # prints: context set to 'bwd.reference.2027'
%catalog get_context()              # -> 'bwd.reference.2027'
%catalog list(".")   # '.' alone -> the context itself: same as list("bwd.reference.2027") -> []

### Going up: `../`

One or more leading `../` (or a bare `..`) walks up that many levels of the context
first, then resolves whatever's left against the result — this works for every relative
path except `use`'s own (always a whole, absolute path — see above).

In [ ]:
%catalog set_tags("../water_temperature", ["archived"])  # up one level, then a sibling — any verb
%catalog get_context()    # -> 'bwd.reference.2027' — unchanged; set_tags only reads context, never writes it

## Read-only queries: `get_wiki`, `get_tags`, `list`, `schema`

These answer immediately too, like `get_context()` above — nothing to commit or undo.
All four raise `CatalogOperationError` if `path` doesn't exist; `schema` also requires a
table or view, not a folder. `list`'s `path` may also be omitted (or `""`) to list the
current context itself — raises `CatalogSessionError` if none is set yet.

In [ ]:
%catalog set_wiki("bwd.reference.water_temperature", "# Water temperature\n\nBathing water assessments.")
%catalog get_wiki("bwd.reference.water_temperature")

In [ ]:
%catalog get_tags("bwd.reference.water_temperature")   # -> ['archived'] — the last set_tags() call above

In [ ]:
%catalog use("bwd.reference")                          # prints: context set to 'bwd.reference'
%catalog list("bwd.reference")                          # every table/view under bwd.reference
%catalog list()                                        # same thing — path omitted, lists the context itself
%catalog schema("bwd.reference.water_temperature")     # column types + row count, no rows fetched

## What a failed call looks like

If a call fails, `%catalog` prints a short message instead of a full traceback (see the
design doc's "Exceptions translated at the boundary") — for example, calling `data_copy`
against a target that already exists without `overwrite=True`:

```
%catalog data_copy("bwd.draft.raw_2026", "bwd.reference.water_temperature")
# -> catalog error: commit failed at step 1/1 (data_copy 'bwd.draft.raw_2026' -> ...):
#    target 'bwd.reference.water_temperature' already exists — pass overwrite=True ...
#    Everything before it was rolled back.
```

That's still `CatalogCommitError` underneath (each call is committed as its own
one-step batch — see `CatalogSession.commit`'s docstring), so the same "rolled back, or
explicit about what it couldn't undo" guarantee applies as before — it just never needs a
separate `%catalog commit` to trigger it. `delete_folder`, `delete_view`/`delete_table`, and
`data_copy`/`data_move` with `overwrite=True`, still can't be undone — see "Must be
all-or-nothing" in the design doc.